In [5]:
# pip install -U langchain langchain-core langchain-community langchain-text-splitters pypdf beautifulsoup4 tiktoken requests

from __future__ import annotations

import re
from pathlib import Path
from typing import List, Optional, Literal
from urllib.parse import urlparse

import requests
from bs4 import Tag
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    HTMLHeaderTextSplitter,
    HTMLSemanticPreservingSplitter,
)

SourceMode = Literal["pdf", "web"]


class HybridRAGChunker:
    def __init__(
        self,
        chunk_size: int = 900,
        chunk_overlap: int = 120,
        model_name_for_tokens: str = "gpt-4o-mini",
    ):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.headers_to_split_on = [
            ("h1", "h1"),
            ("h2", "h2"),
            ("h3", "h3"),
            ("h4", "h4"),
        ]

        self.recursive_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            model_name=model_name_for_tokens,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[
                "\n\n",
                "\n",
                ". ",
                "! ",
                "? ",
                "; ",
                ", ",
                " ",
                "",
            ],
        )

    def _clean_text(self, text: str) -> str:
        text = text.replace("\xa0", " ")
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        return text.strip()

    def _code_handler(self, element: Tag) -> str:
        lang = element.get("data-lang") or "text"
        body = element.get_text(" ", strip=True)
        return f"<code:{lang}>{body}</code>"

    def _enrich_chunks(self, docs: List[Document], source_type: str) -> List[Document]:
        out = []
        total = len(docs)

        for i, d in enumerate(docs):
            content = self._clean_text(d.page_content)
            if not content:
                continue

            meta = dict(d.metadata)
            meta["chunk_index"] = i
            meta["chunk_count"] = total
            meta["char_length"] = len(content)
            meta["source_type"] = source_type
            out.append(Document(page_content=content, metadata=meta))

        return out

    def chunk_pdf(
        self,
        pdf_path: str | Path,
        extraction_mode: Literal["plain", "layout"] = "layout",
        pdf_mode: Literal["page", "single"] = "page",
    ) -> List[Document]:
        loader = PyPDFLoader(
            file_path=str(pdf_path),
            mode=pdf_mode,
            extraction_mode=extraction_mode,
        )
        docs = loader.load()

        for d in docs:
            d.page_content = self._clean_text(d.page_content)
            d.metadata["source"] = str(pdf_path)
            d.metadata["filename"] = Path(pdf_path).name

        split_docs = self.recursive_splitter.split_documents(docs)
        return self._enrich_chunks(split_docs, source_type="pdf")

    def chunk_web(
        self,
        url: str,
        strategy: Literal["semantic", "headers"] = "semantic",
    ) -> List[Document]:
        if strategy == "headers":
            splitter = HTMLHeaderTextSplitter(
                headers_to_split_on=self.headers_to_split_on,
                return_each_element=False,
            )
            docs = splitter.split_text_from_url(url)
            for d in docs:
                d.metadata["source"] = url
                d.metadata["domain"] = urlparse(url).netloc
            split_docs = self.recursive_splitter.split_documents(docs)
            return self._enrich_chunks(split_docs, source_type="web")

        html = requests.get(
            url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0 RAGChunker/1.0"},
        )
        html.raise_for_status()

        splitter = HTMLSemanticPreservingSplitter(
            headers_to_split_on=self.headers_to_split_on,
            max_chunk_size=self.chunk_size,
            separators=["\n\n", "\n", ". ", "! ", "? ", "; "],
            elements_to_preserve=["table", "ul", "ol", "pre", "code"],
            denylist_tags=["script", "style", "head", "nav", "footer"],
            preserve_images=False,
            preserve_videos=False,
            custom_handlers={"code": self._code_handler, "pre": self._code_handler},
        )

        docs = splitter.split_text(html.text)

        for d in docs:
            d.page_content = self._clean_text(d.page_content)
            d.metadata["source"] = url
            d.metadata["domain"] = urlparse(url).netloc

        return self._enrich_chunks(docs, source_type="web")

    def chunk_sources(
        self,
        pdf_paths: Optional[List[str]] = None,
        urls: Optional[List[str]] = None,
    ) -> List[Document]:
        all_chunks: List[Document] = []

        for pdf in pdf_paths or []:
            all_chunks.extend(self.chunk_pdf(pdf))

        for url in urls or []:
            all_chunks.extend(self.chunk_web(url, strategy="semantic"))

        return all_chunks

C:\Users\mahen\AppData\Local\Temp\ipykernel_10664\524735953.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from pathlib import Path
BASE_DIR = Path.cwd().resolve().parent
file_path2 = BASE_DIR / "config" / "data" / "processed" / "Attention.pdf"

In [7]:
file_path2

WindowsPath('D:/AI-Restart-2026/Complete GEN AI Journey - Notes/DataIngestion_Pipeline/config/data/processed/Attention.pdf')

In [10]:
chunker = HybridRAGChunker(chunk_size=900, chunk_overlap=120)

pdf_chunks = chunker.chunk_pdf(
    pdf_path=file_path2,
    extraction_mode="layout",   # good for preserving visual reading order
    pdf_mode="page"
)

pdf_chunks[0].page_content


'Attention Is All You Need\n\n Ashish Vaswani∗ Noam Shazeer∗ Niki Parmar∗ Jakob Uszkoreit∗\n Google Brain Google Brain Google Research Google Research\n avaswani@google.com noam@google.com nikip@google.com usz@google.com\n\n Llion Jones∗ Aidan N. Gomez∗† Łukasz Kaiser∗\n Google Research University of Toronto Google Brain\n llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com\n\n Illia Polosukhin∗‡\n illia.polosukhin@gmail.com\n\n Abstract\n\n The dominant sequence transduction models are based on complex recurrent or\n convolutional neural networks that include an encoder and a decoder. The best\n performing models also connect the encoder and decoder through an attention\n mechanism. We propose a new simple network architecture, the Transformer,\n basedsolelyonattentionmechanisms,dispensingwithrecurrenceandconvolutions\n entirely. Experiments on two machine translation tasks show these models to\n be superior in quality while being moreparallelizable and requiring signiﬁcantl

In [5]:
web_chunks = chunker.chunk_web(
    url="https://www.geeksforgeeks.org/artificial-intelligence/ml-attention-mechanism/",
    strategy="semantic"
)

print(web_chunks[0].metadata)
print(web_chunks[0].page_content[:500])

{'source': 'https://www.geeksforgeeks.org/artificial-intelligence/ml-attention-mechanism/', 'domain': 'www.geeksforgeeks.org', 'chunk_index': 0, 'chunk_count': 19, 'char_length': 1766, 'source_type': 'web'}
Courses Tutorials Practice Jobs Artificial Intelligence Interview Questions Project Ideas Search Algorithms Local Search Algorithm Generative AI Data Science Machine Learning Deep Learning ML-Projects Robotics Share Your Experiences Introduction to AI What is Artificial Intelligence (AI) Types of Artificial Intelligence (AI) Types of AI Based on Functionalities Agents in AI Artificial intelligence vs Machine Learning vs Deep Learning Problem Solving in Artificial Intelligence Top 20 Applications


C:\Users\mahen\AppData\Local\Temp\ipykernel_7916\524735953.py:130: LangChainBetaWarning: The class `HTMLSemanticPreservingSplitter` is in beta. It is actively being worked on, so the API may change.
  splitter = HTMLSemanticPreservingSplitter(


In [6]:
web_chunks

[Document(metadata={'source': 'https://www.geeksforgeeks.org/artificial-intelligence/ml-attention-mechanism/', 'domain': 'www.geeksforgeeks.org', 'chunk_index': 0, 'chunk_count': 19, 'char_length': 1766, 'source_type': 'web'}, page_content='Courses Tutorials Practice Jobs Artificial Intelligence Interview Questions Project Ideas Search Algorithms Local Search Algorithm Generative AI Data Science Machine Learning Deep Learning ML-Projects Robotics Share Your Experiences Introduction to AI What is Artificial Intelligence (AI) Types of Artificial Intelligence (AI) Types of AI Based on Functionalities Agents in AI Artificial intelligence vs Machine Learning vs Deep Learning Problem Solving in Artificial Intelligence Top 20 Applications of Artificial Intelligence (AI) in 2025 AI Concepts Search Algorithms in AI Local Search Algorithm in Artificial Intelligence Adversarial Search Algorithms in Artificial Intelligence (AI) Constraint Satisfaction Problems (CSP) in Artificial Intelligence Know

In [1]:
document_chunks = [
    "Attention mechanisms allow deep learning architectures to dynamically compute token weight priorities.",
    "Bypassing external microservices via native local pipelines drops data transmission overhead to zero.",
    "Vector indices serve as the fundamental indexing system for enterprise Retrieval-Augmented Generation."
]

In [ ]:
pdf_upload_directory = BASE_DIR / "config" / "data" / "uploads" / "Neural-Network(Basics).pdf"